# Can Small Fine-Tuned Models match the  quality of Large LLMs for Mental Health Support?

## Imports and Installations

In [ ]:
from huggingface_hub import login

# Login with token directly
login(token="YOUR_HUGGINGFACE_TOKEN")

In [ ]:
%pip install \
    datasets \
    evaluate \
    rouge_score\
    loralib \
    evaluate \
    accelerate \
    bitsandbytes \
    trl \
    peft \
    -U --quiet

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, GenerationConfig, TrainingArguments, Trainer
import torch
import time
import pandas as pd
import re
import numpy as np
import string
from nltk.corpus import stopwords
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfTransformer,TfidfVectorizer
from sklearn.pipeline import Pipeline
import evaluate

In [ ]:
import os
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    HfArgumentParser,
    TrainingArguments,
    pipeline,
    logging,
)
from peft import LoraConfig, PeftModel
from trl import SFTTrainer

In [ ]:
import transformers

torch.backends.cuda.enable_mem_efficient_sdp(False)
torch.backends.cuda.enable_flash_sdp(False)


## Testing Base model response

In [ ]:
# Checking the base model response
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3-8B-Instruct")
model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Meta-Llama-3-8B-Instruct",
    torch_dtype=torch.float16,  # Use float16 for efficiency
    device_map="auto"  # Automatically handle device placement
)

# Your test question
question = """I feel like I don't exist and my body is not my own,
like if I'm sombody else observin me, what could be this disorder?"""

# Prepare the input
messages = [
    {"role": "user", "content": question}
]

# Tokenize the input
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# Generate response
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=512,
    temperature=0.01,
    top_p=0.9,
    do_sample=True
)

# Decode and print the response
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]
response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

print("Question:", question)
print("\n" + "="*50 + "\n")
print("Response:", response)

## Data Processing and tokenizing

In [ ]:
from transformers import AutoTokenizer


tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3-8B-Instruct")

compute_dtype = getattr(torch, "float16")

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

# Load base model
model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Meta-Llama-3-8B-Instruct",
    quantization_config=quant_config,
    device_map={"": 0}
)
model.config.use_cache = False
model.config.pretraining_tp = 1

In [ ]:
# Set pad_token as end-of-sentence token
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Explicitly set the model's pad_token_id and eos_token_id based on the tokenizer
model.config.pad_token_id = tokenizer.pad_token_id
model.config.eos_token_id = tokenizer.eos_token_id

In [ ]:
def print_number_of_trainable_model_parameters(model):
    trainable_model_params = 0
    all_model_params = 0
    for _, param in model.named_parameters():
        all_model_params += param.numel()
        if param.requires_grad:
            trainable_model_params += param.numel()
    return f"trainable model parameters: {trainable_model_params}\nall model parameters: {all_model_params}\npercentage of trainable model parameters: {100 * trainable_model_params / all_model_params:.2f}%"

print(print_number_of_trainable_model_parameters(model))

In [ ]:
from datasets import load_dataset

dataset1 = load_dataset("Amod/mental_health_counseling_conversations")
dataset2 = load_dataset("mpingale/mental-health-chat-dataset")
dataset3 = load_dataset("heliosbrahma/mental_health_chatbot_dataset")



In [ ]:
df1= dataset1["train"].data.to_pandas().dropna()
df2= dataset2["train"].data.to_pandas().dropna()
df3= dataset3["train"].data.to_pandas().dropna()

In [ ]:
df1.head()

In [ ]:
df1.columns = ["question", "answer"]
df1

In [ ]:
df2.head()

In [ ]:
# Just rename columns
df2 = df2[["questionText", "answerText"]]
df2.columns = ["question", "answer"]
df2

In [ ]:
df3.head()

In [ ]:
def split_qa(text):
    # below split returns ["", "question\n", "answer "]
    question, answer = re.split(r" ?<\w+>: ?", text)[1:]
    question = question.strip().replace("\n","")
    answer = answer.strip().replace("\n", "")
    return question, answer

df3["text"] = df3["text"].apply(split_qa)
df3[["question", "answer"]] = pd.DataFrame(data=[[q, a] for q, a in df3["text"].values])
df3.drop(columns=["text"], inplace=True)
df3

In [ ]:
# Merge datasets
df = pd.concat((df1, df2, df3)).sample(frac=1)
df


In [ ]:
df, test_df = train_test_split(df, test_size=0.016)

In [ ]:
df

In [ ]:
test_df

In [ ]:
test_df.to_csv('test_df.csv', index=False)

In [ ]:
def tokenize_function(row):
    # Tokenize the conversations
    row['input_ids'] = tokenizer(row["question"], padding="max_length", truncation=True, max_length = 128, return_tensors="pt").input_ids[0]

    # Assuming "answer" column is already a string, no need for conversion
    row['labels'] = tokenizer(row["answer"], padding="max_length", truncation=True, max_length = 128, return_tensors="pt").input_ids[0]

    return row


# Tokenize the DataFrame
tokenized_df = df.apply(tokenize_function, axis=1)
tokenized_df['input_ids'] = tokenized_df['input_ids'].apply(lambda x: x.tolist())
tokenized_df['labels'] = tokenized_df['labels'].apply(lambda x: x.tolist())

In [ ]:
tokenized_df

In [ ]:
# Remove texts and keep input_ids and labels
tokenized_df.drop(columns=["question", "answer"], inplace=True)

## QLORA fine tuning and saving the model

In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(tokenized_df)

In [ ]:


# Load LoRA configuration
peft_args = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.1,
    r=8,
    bias="none",
    task_type="CAUSAL_LM",
)



In [ ]:
# Set training parameters
training_params = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=1,
#     evaluation_strategy="epoch",
    optim="paged_adamw_32bit",
    save_steps=500,
    logging_steps=100,
    learning_rate=6e-5,
    weight_decay=0.001,
    fp16=True,
    bf16=False,
    max_grad_norm=0.3,
#     max_steps=-1,
    warmup_ratio=0.03,
    group_by_length=True,
    lr_scheduler_type="constant",
    report_to="tensorboard"
)

In [ ]:
from trl import SFTConfig
sft_config = SFTConfig(
    output_dir="./results",
    dataset_text_field="text",  # Specify the text field name
    # max_seq_length=None,  # or your desired length
    packing=False,
    report_to="none"
    # ... other training arguments
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    peft_config=peft_args,
    processing_class=tokenizer,
    args=sft_config,  # Use SFTConfig instead of TrainingArguments
)

In [ ]:
trainer.train()

In [ ]:
trainer.model.save_pretrained("./llama3-8B-Psychotherapist")
tokenizer.save_pretrained("./llama3-8B-Psychotherapist")



In [ ]:
import gc
del model
gc.collect()
torch.cuda.empty_cache()

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    "./llama3-8B-Psychotherapist",  # Your saved model path
    quantization_config=quant_config,
    device_map={"": 0}
)

# IMPORTANT: Don't set use_cache = False this time!
# model.config.use_cache = False  # <-- DON'T DO THIS
model.config.pretraining_tp = 1

# Set pad_token as end-of-sentence token
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Explicitly set the model's pad_token_id and eos_token_id
model.config.pad_token_id = tokenizer.pad_token_id
model.config.eos_token_id = tokenizer.eos_token_id

# Ensure model is in eval mode and gradient checkpointing is disabled
model.eval()

In [ ]:
print("Merging LoRA weights...")
merged_model = trainer.model.merge_and_unload()

# Save the merged model
print("Saving merged model...")
merged_model.save_pretrained("./llama3-8B-Psychotherapist-merged")
tokenizer.save_pretrained("./llama3-8B-Psychotherapist-merged")

In [ ]:
del model
del trainer
gc.collect()
torch.cuda.empty_cache()

In [ ]:
print("Reloading merged model...")
model = AutoModelForCausalLM.from_pretrained(
    "./llama3-8B-Psychotherapist-merged",
    quantization_config=quant_config,
    device_map={"": 0}
)

In [ ]:
model.config.pretraining_tp = 1
model.eval()

In [ ]:
def single_inference(question):
    messages = [
        {"role": "system", "content": "Reply to the following inquiry as a pyschotherapist."},
        {"role": "user", "content": question}
    ]

    model.config.use_cache = True

    # Disable gradient checkpointing for inference
    if hasattr(model, 'gradient_checkpointing_disable'):
        model.gradient_checkpointing_disable()

    # Ensure model is in eval mode
    model.eval()

    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)

    # Create attention mask
    attention_mask = torch.ones_like(input_ids)

    # Handle terminators
    terminators = []
    if tokenizer.eos_token_id is not None:
        terminators.append(tokenizer.eos_token_id)

    eot_token_id = tokenizer.convert_tokens_to_ids("<|eot_id|>")
    if eot_token_id is not None and eot_token_id != tokenizer.unk_token_id:
        terminators.append(eot_token_id)

    generate_kwargs = {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "max_new_tokens": 256,
        "do_sample": True,
        "temperature": 0.01,
        "pad_token_id": tokenizer.pad_token_id or tokenizer.eos_token_id,
    }

    if terminators:
        generate_kwargs["eos_token_id"] = terminators

    # Use no_grad context for inference
    with torch.no_grad():
        outputs = model.generate(**generate_kwargs)

    response = outputs[0][input_ids.shape[-1]:]
    output = tokenizer.decode(response, skip_special_tokens=True)
    return output



In [ ]:
question = """I feel like I don't exist and my body is not my own,
like if I'm sombody else observin me, what could be this disorder?"""

answer = single_inference(question)

print(f'INPUT QUESTION:\n{question}')
print(f'\n\nModel Answer:\n{answer}')

In [ ]:
print(f"Model dtype: {next(model.parameters()).dtype}")
print(f"Model device: {next(model.parameters()).device}")

In [ ]:
question = """I'm always confused when making decisions, I cannot choose one option,
I must have only one option so I can make a decision, what are the causes of being
indecisive?"""

answer = single_inference(question)

print(f'INPUT QUESTION:\n{question}')
print(f'\n\nModel Answer:\n{answer}')

### Generating answers for the test set from the QLORA fine Tuned model

In [ ]:
import pandas as pd

# Load the test data
df = pd.read_csv('test_df.csv')

# Generate answers for all questions
answers = []
for idx, row in df.iterrows():
    question = row['question']
    print(f"Processing question {idx + 1}/{len(df)}...")

    try:
        answer = single_inference(question)
        answers.append(answer)
    except Exception as e:
        print(f"Error processing question {idx}: {e}")
        answers.append("")

# Create results dataframe
results_df = pd.DataFrame({
    'question': df['question'],
    'answer': answers
})

# Save to CSV
results_df.to_csv('fine_tuned_model_answers.csv', index=False)
print(f"\nAnswers saved to fine_tuned_model_answers.csv")

## Invoke Gemini client and function to get LLM response

In [ ]:
GEMINI_API_KEY = "GEMINI_API_KEY"

In [ ]:
from google import genai
from google.genai import types
import os

# Initialize the client
client = genai.Client(api_key=GEMINI_API_KEY)
# client = genai.Client(api_key=api_key)
def get_llm_response(prompt):
    try:
        response = client.models.generate_content(
            model="gemini-2.5-flash", # Or specific version like "gemini-2.5-flash-preview-09-2025"
            contents=prompt,
            config=types.GenerateContentConfig(
                temperature=0.1, # Low temperature for more deterministic/factual responses
                top_p=0.95,
            )
        )
        return response.text

    except Exception as e:
        print(f"An error occurred: {e}")

In [ ]:
question = """I feel like I don't exist and my body is not my own,
like if I'm sombody else observin me, what could be this disorder?
response should be around 300 words """

answer = get_llm_response(question)

print(f'INPUT QUESTION:\n{question}')
print(f'\n\nModel Answer:\n{answer}')

## Getting the response for zero-shot Prompting, COT Prompting, Role Prompting

### Generating and saving Zero-shot Prompting Responses

In [ ]:
zero_shot_prompt = """
{question}
response should be around 300 words
"""

In [ ]:
import pandas as pd

df = pd.read_csv('sample_data/test_df.csv')

# Generate answers for all questions
answers = []
for idx, row in df.iterrows():
    question = row['question']
    print(f"Processing question {idx + 1}/{len(df)}...")

    try:
        answer = get_llm_response(zero_shot_prompt.format(question=question))
        answers.append(answer)
    except Exception as e:
        print(f"Error processing question {idx}: {e}")
        answers.append("")

# Create results dataframe
results_df = pd.DataFrame({
    'question': df['question'],
    'answer': answers
})

# Save to CSV
results_df.to_csv('zero_shot_prompt_answers.csv', index=False)
print(f"\nAnswers saved to fine_tuned_model_answers.csv")

### Generating and saving COT Prompting Responses

In [ ]:
cot_prompting = """

When a client shares their concerns with you, work through the following reasoning process step-by-step before responding. think through process of ariving at the response, then provide your counseling response.

Chain of Thought Steps:

Step 1: Identify Core Emotions
Think through: What emotions is the client expressing directly? What emotions might be underlying or unspoken?

Step 2: Analyze the Situation
Think through: What is the main issue? What are the contextual factors? What relationship dynamics exist? Are there safety concerns?

Step 3: Assess Client Needs
Think through: What does this person need most right now - validation, coping strategies, perspective, communication guidance, or something else?

Step 4: Select Therapeutic Approach
Think through: Which counseling techniques would be most helpful - reflective listening, cognitive reframing, emotion regulation, boundary-setting guidance?

Step 5: Formulate Response
Think through: What key elements should my response include? Consider structure, tone, content balance (emotional support vs. practical advice), and how to encourage client autonomy and continued engagement.

client question:
{question}

response should be around 300 words
"""

In [ ]:
import pandas as pd

df = pd.read_csv('sample_data/test_df.csv')

# Generate answers for all questions
answers = []
for idx, row in df.iterrows():
    question = row['question']
    print(f"Processing question {idx + 1}/{len(df)}...")

    try:
        answer = get_llm_response(cot_prompting.format(question=question))
        answers.append(answer)
    except Exception as e:
        print(f"Error processing question {idx}: {e}")
        answers.append("")

# Create results dataframe
results_df = pd.DataFrame({
    'question': df['question'],
    'answer': answers
})

# Save to CSV
results_df.to_csv('cot_prompt_answers.csv', index=False)
print(f"\nAnswers saved to fine_tuned_model_answers.csv")

### Generating and saving Role Prompting Responses

In [ ]:
role_prompting = """

You are a Doctor, a licensed professional counselor with 12 years of experience specializing in relationship counseling and emotional wellness. You have a warm, empathetic communication style and use evidence-based therapeutic approaches including Cognitive Behavioral Therapy (CBT), Emotionally Focused Therapy (EFT), and Person-Centered Therapy.


### Your Counseling Approach:
- **Empathy First**: You always validate emotions before offering perspective
- **Client-Centered**: You empower clients to find their own answers rather than prescribing solutions
- **Strength-Based**: You help clients recognize their resilience and coping abilities
- **Practical**: You provide concrete strategies alongside emotional support
- **Culturally Sensitive**: You respect diverse backgrounds and relationship structures

### Your Communication Style:
- Warm and genuine, like talking to a trusted mentor
- Clear and direct without being harsh
- You normalize common human experiences
- You ask thoughtful questions to deepen understanding
- You balance validation with gentle challenges when appropriate
- You use "I" statements to share observations ("I notice...", "I'm hearing...")

### Your Boundaries:
- You provide guidance and support, not diagnoses or prescriptions
- You acknowledge when issues are beyond your scope
- You maintain confidentiality and professionalism

client question:
{question}

response should be around 300 words
"""

In [ ]:
import pandas as pd

df = pd.read_csv('sample_data/test_df.csv')

# Generate answers for all questions
answers = []
for idx, row in df.iterrows():
    question = row['question']
    print(f"Processing question {idx + 1}/{len(df)}...")

    try:
        answer = get_llm_response(role_prompting.format(question=question))
        answers.append(answer)
    except Exception as e:
        print(f"Error processing question {idx}: {e}")
        answers.append("")

# Create results dataframe
results_df = pd.DataFrame({
    'question': df['question'],
    'answer': answers
})

# Save to CSV
results_df.to_csv('role_prompt_answers.csv', index=False)
print(f"\nAnswers saved to fine_tuned_model_answers.csv")

In [ ]:
# Answer regenerated for the questions that LLM failed to respond due to free token limits
import pandas as pd

# Load the existing file
df = pd.read_csv('sample_data/role_prompt_answers.csv')

# Loop over rows and fill empty answers
for idx, row in df.iterrows():
    if pd.isna(row['answer']) or str(row['answer']).strip() == "":
        question = row['question']
        print(f"Empty answer found at row {idx}. Generating new answer...")

        try:
            new_answer = get_llm_response(role_prompting.format(question=question))
        except Exception as e:
            print(f"Error generating answer at row {idx}: {e}")
            new_answer = ""

        df.at[idx, 'answer'] = new_answer

# Save back to the same file
df.to_csv('sample_data/role_prompt_answers.csv', index=False)

print("Updated empty answers and saved back to role_prompt_answers.csv")


### Cleaning the COT prompting response

In [ ]:
import pandas as pd

file_path = "cleaned_cot_prompt_answers.csv"   # <-- change this

# Read file
df = pd.read_csv(file_path)

# Loop through rows
for idx, row in df.iterrows():
    answer = str(row["answer"])  # ensure string
    keyword = "**Step 5: Formulate Response**"

    if keyword in answer:
        # Extract part after the keyword
        new_answer = answer.split(keyword, 1)[1].strip()

        # Replace in dataframe
        df.at[idx, "answer"] = new_answer

# Save cleaned file (overwrite or new file)
df.to_csv("cleaned_cot_prompt_answers.csv", index=False)

print("Cleaning completed. Saved as:", "cleaned_cot_prompt_answers.csv" + file_path)


## LLM as a Judge used to evaluate all the 4 approaches
### 1. Fine tuned model response
### 2. Zero shot prompted response
### 3. COT prompted response
### 4. Role prompted response

In [ ]:
llm_as_a_judge = """
You are an expert evaluator assessing mental health counseling responses. Your task is to evaluate multiple responses to the same client inquiry.

QUESTION: {question}

YOUR TASK: Evaluate each response on the following dimensions using a 1-10 scale, where:

    1-3 = Poor/Inadequate
    4-6 = Adequate/Satisfactory
    7-8 = Good/Strong
    9-10 = Excellent/Exceptional

EVALUATION CRITERIA:

    Empathy and Validation (Weight: 20%)
        Does the response acknowledge and validate the client's emotions?
        Is there genuine warmth and understanding conveyed?
        Does it normalize the client's experience appropriately?
        Are the client's feelings treated as legitimate and understandable?
    Clinical Appropriateness and Safety (Weight: 20%)
        Is the response emotionally safe and non-judgmental?
        Does it avoid harmful assumptions, advice, or pathologizing?
        Are boundaries appropriate for the therapeutic context?
        Does it recognize when professional intervention is needed?
        Does it avoid overpromising or giving guarantees about outcomes?
    Therapeutic Depth and Insight (Weight: 15%)
        Does the response go beyond surface-level comfort?
        Are there meaningful observations about the client's situation?
        Does it help the client gain perspective or self-understanding?
        Does it identify underlying patterns or themes when appropriate?
    Cultural Competence and Sensitivity (Weight: 15%)
        Is the response culturally aware and sensitive?
        Does it avoid stereotypes or cultural assumptions?
        Is it affirming of diverse identities and experiences?
        Does it consider contextual factors (social, cultural, systemic)?
    Comprehensiveness (Weight: 10%)
        Does it address all key issues presented by the client?
        Are multiple dimensions of the problem acknowledged?
        Does it recognize the complexity of the situation?
        Are interconnected issues identified and addressed?
    Actionability and Guidance (Weight: 10%)
        Does it provide clear, practical next steps or suggestions?
        Are recommendations appropriate and achievable?
        Does it encourage further exploration or professional support?
        Does it empower the client rather than prescribe solutions?
    Engagement and Therapeutic Alliance (Weight: 5%)
        Does the response invite continued dialogue?
        Does it create a sense of being heard and understood?
        Would this response encourage the client to continue opening up?
        Does it establish or maintain rapport?
    Language Quality and Communication (Weight: 5%)
        Is the language clear, compassionate, and appropriate?
        Are there any awkward phrasings, jargon, or redundancies?
        Does it maintain professional standards while being warm?
        Is the tone consistent and appropriate throughout?

OUTPUT FORMAT:

For each response, provide:

Scores:

Criterion	Score (1-10)	Weighted Score
Empathy and Validation (20%)	X	X.XX
Clinical Appropriateness and Safety (20%)	X	X.XX
Therapeutic Depth and Insight (15%)	X	X.XX
Cultural Competence and Sensitivity (15%)	X	X.XX
Comprehensiveness (10%)	X	X.XX
Actionability and Guidance (10%)	X	X.XX
Engagement and Therapeutic Alliance (5%)	X	X.XX
Language Quality and Communication (5%)	X	X.XX
TOTAL WEIGHTED SCORE		X.XX/10

Notable Observations: [Any particularly effective or problematic elements, unique approaches, or standout features]

Overall Assessment: [2-3 sentence summary of the response's quality and appropriateness]

COMPARATIVE ANALYSIS:

Final Ranking:

    [Response X] - [Brief justification]
    [Response Y] - [Brief justification]
    [Response Z] - [Brief justification]
    [Response W] - [Brief justification]

Key Differentiators: [What made the top responses stand out? What were critical weaknesses in lower-ranked responses?]

Recommendations: [General insights about what makes an effective response to this type of client concern]

THE CLIENT QUESTION: {question}

THE RESPONSES TO EVALUATE:

Response 1: {answer_1}

Response 2: {answer_2}

Response 3: {answer_3}

Response 4: {answer_4}
"""

In [ ]:
import pandas as pd


file_paths = [
    "sample_data/fine_tuned_model_answers.csv",
    "sample_data/zero_shot_prompt_answers.csv",
    "sample_data/cleaned_cot_prompt_answers.csv",
    "sample_data/role_prompt_answers.csv"
]

dfs = [pd.read_csv(fp) for fp in file_paths]

# ---- Combine into One DataFrame ----
combined = pd.DataFrame()
combined["question"] = dfs[0]["question"]

for i, df in enumerate(dfs):
    combined[f"answer_{i+1}"] = df["answer"]

# ---- Loop & Print ----
results = []

for idx, row in combined.iterrows():
    print(f"Processing row {idx + 1}/{len(combined)}...")
    question = row["question"]
    llm_as_a_judge_prompt = llm_as_a_judge.format(question=question,
                                                  answer_1=row["answer_1"],
                                                  answer_2=row["answer_2"],
                                                  answer_3=row["answer_3"],
                                                  answer_4=row["answer_4"])
    # LLM Call
    llm_result = get_llm_response(llm_as_a_judge_prompt)

    # Store for output file
    results.append({
        "question": question,
        "fine_tuned_ans": row["answer_1"],
        "zero_shot_ans": row["answer_2"],
        "cot_ans": row["answer_3"],
        "role_prompting_ans": row["answer_4"],
        "llm_result": llm_result
    })

out_df = pd.DataFrame(results)
out_df.to_csv("llm_as_a_judge.csv", index=False)

print("\nSaved: llm_as_a_judge.csv")


In [ ]:
import re
import pandas as pd

# Load your CSV
df = pd.read_csv("sample_data/llm_as_a_judge.csv")

# Regex pattern (extract scores like 5.50/10)
pattern = r"TOTAL WEIGHTED SCORE(?:\*\*)?.*?(\d+\.\d+)/10"


# Create empty columns first
for i in range(1, 5):
    df[f"ans_{i}_score"] = None

# Loop through each row
for idx, row in df.iterrows():
    text = row["llm_result"]

    # Extract all matching scores from this row
    scores = re.findall(pattern, text)
    print(scores)
    # Ensure exactly 4 scores exist; if not, fill missing with None
    while len(scores) < 4:
        scores.append(None)

    # Assign the scores to new columns
    for i in range(4):
        df.at[idx, f"ans_{i+1}_score"] = float(scores[i]) if scores[i] else None

# Save the updated CSV
df.to_csv("llm_as_a_judge.csv", index=False)

print("Done! Score columns added.")


In [ ]:
import pandas as pd

# Read the CSV
df = pd.read_csv("llm_as_a_judge.csv")

cols = ["ans_1_score", "ans_2_score", "ans_3_score", "ans_4_score"]

# Compute total sum of each column
column_sums = df[cols].sum()

print("Total sum of each column:")
print(column_sums)

In [ ]:
import pandas as pd
import re
import json

def parse_evaluation_text(text):
    """
    Parse evaluation text to extract scores for each response.
    Returns a dictionary with response numbers as keys.
    """
    results = {}

    # Split text by response sections
    response_sections = re.split(r'\*\*Response (\d+):\*\*', text)

    # Process each response (starts at index 1, pairs of [number, content])
    for i in range(1, len(response_sections), 2):
        response_num = response_sections[i]
        response_content = response_sections[i + 1]

        # Define criteria to extract
        criteria = {
            "Empathy and Validation": 0,
            "Clinical Appropriateness and Safety": 0,
            "Therapeutic Depth and Insight": 0,
            "Cultural Competence and Sensitivity": 0,
            "Comprehensiveness": 0,
            "Actionability and Guidance": 0,
            "Engagement and Therapeutic Alliance": 0,
            "Language Quality and Communication": 0
        }

        # Extract scores using regex patterns
        for criterion in criteria.keys():
            # Pattern to match the criterion and its score
            pattern = rf'{re.escape(criterion)}.*?(\d+)\s+[\d.]+'
            match = re.search(pattern, response_content)
            if match:
                criteria[criterion] = int(match.group(1))

        results[f"response{response_num}"] = criteria

    return results

def process_csv_to_json(csv_file, output_file='output.json'):
    """
    Read CSV file, process llm_result column, and aggregate scores.
    """
    # Read CSV
    df = pd.read_csv(csv_file)

    # Initialize aggregated results
    aggregated = {
        "response1": {},
        "response2": {},
        "response3": {},
        "response4": {}
    }

    criteria_list = [
        "Empathy and Validation",
        "Clinical Appropriateness and Safety",
        "Therapeutic Depth and Insight",
        "Cultural Competence and Sensitivity",
        "Comprehensiveness",
        "Actionability and Guidance",
        "Engagement and Therapeutic Alliance",
        "Language Quality and Communication"
    ]

    # Initialize sums
    for response in aggregated.keys():
        for criterion in criteria_list:
            aggregated[response][f"total sum of {criterion}"] = 0

    # Process each row
    for idx, row in df.iterrows():
        llm_result = row['llm_result']

        # Parse the evaluation text
        parsed_results = parse_evaluation_text(llm_result)

        # Aggregate scores
        for response_key, scores in parsed_results.items():
            for criterion, score in scores.items():
                aggregated[response_key][f"total sum of {criterion}"] += score

    # Convert to JSON
    json_output = json.dumps(aggregated, indent=2)

    # Save to file
    with open(output_file, 'w') as f:
        f.write(json_output)

    print(f"Results saved to {output_file}")
    print("\nAggregated Results:")
    print(json_output)

    # Create a summary table
    create_summary_table(aggregated)

    return aggregated

def create_summary_table(aggregated):
    """
    Create a formatted table from the aggregated results.
    """
    # Map response keys to descriptive names
    response_mapping = {
        "response1": "Fine-tuned Model",
        "response2": "Zero Shot Prompt+LLM",
        "response3": "CoT Prompt+LLM",
        "response4": "Role Prompt+LLM"
    }

    # Extract criteria names (remove "total sum of " prefix)
    criteria = list(aggregated["response1"].keys())

    # Create DataFrame with mapped column names
    table_data = {}
    for response_key in ["response1", "response2", "response3", "response4"]:
        mapped_name = response_mapping[response_key]
        table_data[mapped_name] = [
            aggregated[response_key][criterion] for criterion in criteria
        ]

    # Clean up criteria names for row labels
    row_labels = [criterion.replace("total sum of ", "") for criterion in criteria]

    df_table = pd.DataFrame(table_data, index=row_labels)

    # Add total row
    df_table.loc['TOTAL'] = df_table.sum()

    print("\n" + "="*100)
    print("SUMMARY TABLE")
    print("="*100)
    print(df_table.to_string())
    print("="*100)

    # Save table to CSV
    df_table.to_csv('summary_table.csv')
    print("\nTable saved to 'summary_table.csv'")

    return df_table

# Example usage
if __name__ == "__main__":
    # Replace 'your_file.csv' with your actual CSV filename
    csv_filename = 'llm_as_a_judge.csv'

    try:
        results = process_csv_to_json(csv_filename)
    except FileNotFoundError:
        print(f"Error: Could not find '{csv_filename}'")
        print("Please update the filename in the script.")
    except Exception as e:
        print(f"Error processing file: {e}")

## Comparing the performance of base model with the Fine tuned model

In [ ]:
# Checking the base model response
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3-8B-Instruct")
model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Meta-Llama-3-8B-Instruct",
    torch_dtype=torch.float16,  # Use float16 for efficiency
    device_map="auto"  # Automatically handle device placement
)

In [ ]:
def base_model_answer(question):

    # Prepare the input
    messages = [
        {"role": "user", "content": question}
    ]

    # Tokenize the input
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    # Generate response
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=300,
        temperature=0.01,
        top_p=0.9,
        do_sample=True
    )

    # Decode and print the response
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return response
# question = "testing the model"
# response = base_model_answer(question)
# print("Question:", question)
# print("\n" + "="*50 + "\n")
# print("Response:", response)

In [ ]:
import pandas as pd

# Load the test data
df = pd.read_csv('sample_data/test_df.csv')

# Generate answers for all questions
answers = []
for idx, row in df.iterrows():
    question = row['question']
    print(f"Processing question {idx + 1}/{len(df)}...")

    try:
        answer = base_model_answer(question)
        answers.append(answer)
    except Exception as e:
        print(f"Error processing question {idx}: {e}")
        answers.append("")

# Create results dataframe
results_df = pd.DataFrame({
    'question': df['question'],
    'answer': answers
})

# Save to CSV
results_df.to_csv('llama_base_model_answers.csv', index=False)
print(f"\nAnswers saved to fine_tuned_model_answers.csv")

In [ ]:
results_df = pd.DataFrame({
    'question': df['question'][:25],
    'answer': answers
})

In [ ]:
results_df.to_csv('llama_base_model_answers.csv', index=False)
print(f"\nAnswers saved to fine_tuned_model_answers.csv")

In [ ]:
llm_as_a_judge_base_llama = """
You are an expert evaluator assessing mental health counseling responses. Your task is to evaluate multiple responses to the same client inquiry.

QUESTION: {question}

YOUR TASK: Evaluate each response on the following dimensions using a 1-10 scale, where:

    1-3 = Poor/Inadequate
    4-6 = Adequate/Satisfactory
    7-8 = Good/Strong
    9-10 = Excellent/Exceptional

EVALUATION CRITERIA:

    Empathy and Validation (Weight: 20%)
        Does the response acknowledge and validate the client's emotions?
        Is there genuine warmth and understanding conveyed?
        Does it normalize the client's experience appropriately?
        Are the client's feelings treated as legitimate and understandable?
    Clinical Appropriateness and Safety (Weight: 20%)
        Is the response emotionally safe and non-judgmental?
        Does it avoid harmful assumptions, advice, or pathologizing?
        Are boundaries appropriate for the therapeutic context?
        Does it recognize when professional intervention is needed?
        Does it avoid overpromising or giving guarantees about outcomes?
    Therapeutic Depth and Insight (Weight: 15%)
        Does the response go beyond surface-level comfort?
        Are there meaningful observations about the client's situation?
        Does it help the client gain perspective or self-understanding?
        Does it identify underlying patterns or themes when appropriate?
    Cultural Competence and Sensitivity (Weight: 15%)
        Is the response culturally aware and sensitive?
        Does it avoid stereotypes or cultural assumptions?
        Is it affirming of diverse identities and experiences?
        Does it consider contextual factors (social, cultural, systemic)?
    Comprehensiveness (Weight: 10%)
        Does it address all key issues presented by the client?
        Are multiple dimensions of the problem acknowledged?
        Does it recognize the complexity of the situation?
        Are interconnected issues identified and addressed?
    Actionability and Guidance (Weight: 10%)
        Does it provide clear, practical next steps or suggestions?
        Are recommendations appropriate and achievable?
        Does it encourage further exploration or professional support?
        Does it empower the client rather than prescribe solutions?
    Engagement and Therapeutic Alliance (Weight: 5%)
        Does the response invite continued dialogue?
        Does it create a sense of being heard and understood?
        Would this response encourage the client to continue opening up?
        Does it establish or maintain rapport?
    Language Quality and Communication (Weight: 5%)
        Is the language clear, compassionate, and appropriate?
        Are there any awkward phrasings, jargon, or redundancies?
        Does it maintain professional standards while being warm?
        Is the tone consistent and appropriate throughout?

OUTPUT FORMAT:

For each response, provide:

Scores:

Criterion	Score (1-10)	Weighted Score
Empathy and Validation (20%)	X	X.XX
Clinical Appropriateness and Safety (20%)	X	X.XX
Therapeutic Depth and Insight (15%)	X	X.XX
Cultural Competence and Sensitivity (15%)	X	X.XX
Comprehensiveness (10%)	X	X.XX
Actionability and Guidance (10%)	X	X.XX
Engagement and Therapeutic Alliance (5%)	X	X.XX
Language Quality and Communication (5%)	X	X.XX
TOTAL WEIGHTED SCORE		X.XX/10

Notable Observations: [Any particularly effective or problematic elements, unique approaches, or standout features]

Overall Assessment: [2-3 sentence summary of the response's quality and appropriateness]

COMPARATIVE ANALYSIS:

Final Ranking:

    [Response X] - [Brief justification]
    [Response Y] - [Brief justification]
    [Response Z] - [Brief justification]
    [Response W] - [Brief justification]

Key Differentiators: [What made the top responses stand out? What were critical weaknesses in lower-ranked responses?]

Recommendations: [General insights about what makes an effective response to this type of client concern]

THE CLIENT QUESTION: {question}

THE RESPONSES TO EVALUATE:

Response 1: {answer_1}

Response 2: {answer_2}

"""

In [ ]:
import pandas as pd


file_paths = [
    "sample_data/fine_tuned_model_answers.csv",
    "llama_base_model_answers.csv",
]

dfs = [pd.read_csv(fp, nrows=25) for fp in file_paths]

# ---- Combine into One DataFrame ----
combined = pd.DataFrame()
combined["question"] = dfs[0]["question"]

for i, df in enumerate(dfs):
    combined[f"answer_{i+1}"] = df["answer"]

# ---- Loop & Print ----
results = []

for idx, row in combined.iterrows():
    print(f"Processing row {idx + 1}/{len(combined)}...")
    question = row["question"]
    llm_as_a_judge_prompt = llm_as_a_judge_base_llama.format(question=question,
                                                  answer_1=row["answer_1"],
                                                  answer_2=row["answer_2"])
    # LLM Call
    llm_result = get_llm_response(llm_as_a_judge_prompt)

    # Store for output file
    results.append({
        "question": question,
        "fine_tuned_ans": row["answer_1"],
        "base_ans": row["answer_2"],
        "llm_result": llm_result
    })

out_df = pd.DataFrame(results)
out_df.to_csv("llm_as_a_judge_base_llama.csv", index=False)

print("\nSaved: llm_as_a_judge.csv")

In [ ]:
import pandas as pd

# ---- Check if output file exists and load it ----
output_file = "llm_as_a_judge_base_llama.csv"

try:
    out_df = pd.read_csv(output_file)
    print(f"Loaded existing file: {output_file}")

    # Check if llm_result column exists
    if "llm_result" not in out_df.columns:
        print("llm_result column not found. Will process all rows.")
        out_df["llm_result"] = None
    else:
        print(f"Found {out_df['llm_result'].notna().sum()} existing results.")

except FileNotFoundError:
    print(f"{output_file} not found. Creating new file.")

    # ---- Load source files ----
    file_paths = [
        "sample_data/fine_tuned_model_answers.csv",
        "llama_base_model_answers.csv",
    ]

    dfs = [pd.read_csv(fp, nrows=25) for fp in file_paths]

    # ---- Combine into One DataFrame ----
    out_df = pd.DataFrame()
    out_df["question"] = dfs[0]["question"]
    out_df["fine_tuned_ans"] = dfs[0]["answer"]
    out_df["base_ans"] = dfs[1]["answer"]
    out_df["llm_result"] = None

# ---- Process rows that don't have llm_result ----
for idx, row in out_df.iterrows():
    if pd.isna(row["llm_result"]) or row["llm_result"] == "":
        print(f"Processing row {idx + 1}/{len(out_df)}...")

        question = row["question"]
        llm_as_a_judge_prompt = llm_as_a_judge_base_llama.format(
            question=question,
            answer_1=row["fine_tuned_ans"],
            answer_2=row["base_ans"]
        )

        # LLM Call
        llm_result = get_llm_response(llm_as_a_judge_prompt)

        # Update the dataframe
        out_df.at[idx, "llm_result"] = llm_result

        # Save after each row (optional, for safety)
        out_df.to_csv(output_file, index=False)
    else:
        print(f"Row {idx + 1}/{len(out_df)} already has result. Skipping.")

# ---- Final save ----
out_df.to_csv(output_file, index=False)
print(f"\nSaved: {output_file}")

In [ ]:
import pandas as pd
import re
import json

def parse_evaluation_text(text):
    """
    Parse evaluation text to extract scores for each response.
    Returns a dictionary with response numbers as keys.
    """
    results = {}

    # Split text by response sections
    response_sections = re.split(r'\*\*Response (\d+):\*\*', text)

    # Process each response (starts at index 1, pairs of [number, content])
    for i in range(1, len(response_sections), 2):
        response_num = response_sections[i]
        response_content = response_sections[i + 1]

        # Define criteria to extract
        criteria = {
            "Empathy and Validation": 0,
            "Clinical Appropriateness and Safety": 0,
            "Therapeutic Depth and Insight": 0,
            "Cultural Competence and Sensitivity": 0,
            "Comprehensiveness": 0,
            "Actionability and Guidance": 0,
            "Engagement and Therapeutic Alliance": 0,
            "Language Quality and Communication": 0
        }

        # Extract scores using regex patterns
        for criterion in criteria.keys():
            # Pattern to match the criterion and its score
            pattern = rf'{re.escape(criterion)}.*?(\d+)\s+[\d.]+'
            match = re.search(pattern, response_content)
            if match:
                criteria[criterion] = int(match.group(1))

        results[f"response{response_num}"] = criteria

    return results

def process_csv_to_json(csv_file, output_file='output.json'):
    """
    Read CSV file, process llm_result column, and aggregate scores.
    """
    # Read CSV
    df = pd.read_csv(csv_file)

    # Initialize aggregated results
    aggregated = {
        "response1": {},
        "response2": {},
    }

    criteria_list = [
        "Empathy and Validation",
        "Clinical Appropriateness and Safety",
        "Therapeutic Depth and Insight",
        "Cultural Competence and Sensitivity",
        "Comprehensiveness",
        "Actionability and Guidance",
        "Engagement and Therapeutic Alliance",
        "Language Quality and Communication"
    ]

    # Initialize sums
    for response in aggregated.keys():
        for criterion in criteria_list:
            aggregated[response][f"total sum of {criterion}"] = 0

    # Process each row
    for idx, row in df.iterrows():
        llm_result = row['llm_result']

        # Parse the evaluation text
        parsed_results = parse_evaluation_text(llm_result)

        # Aggregate scores
        for response_key, scores in parsed_results.items():
            for criterion, score in scores.items():
                aggregated[response_key][f"total sum of {criterion}"] += score

    # Convert to JSON
    json_output = json.dumps(aggregated, indent=2)

    # Save to file
    with open(output_file, 'w') as f:
        f.write(json_output)

    print(f"Results saved to {output_file}")
    print("\nAggregated Results:")
    print(json_output)

    # Create a summary table
    create_summary_table(aggregated)

    return aggregated

def create_summary_table(aggregated):
    """
    Create a formatted table from the aggregated results.
    """
    # Map response keys to descriptive names
    response_mapping = {
        "response1": "Llama base Model",
        "response2": "Fine-tuned Model",
    }

    # Extract criteria names (remove "total sum of " prefix)
    criteria = list(aggregated["response1"].keys())

    # Create DataFrame with mapped column names
    table_data = {}
    for response_key in ["response1", "response2"]:
        mapped_name = response_mapping[response_key]
        table_data[mapped_name] = [
            aggregated[response_key][criterion] for criterion in criteria
        ]

    # Clean up criteria names for row labels
    row_labels = [criterion.replace("total sum of ", "") for criterion in criteria]

    df_table = pd.DataFrame(table_data, index=row_labels)

    # Add total row
    df_table.loc['TOTAL'] = df_table.sum()

    print("\n" + "="*100)
    print("SUMMARY TABLE")
    print("="*100)
    print(df_table.to_string())
    print("="*100)

    # Save table to CSV
    df_table.to_csv('summary_table.csv')
    print("\nTable saved to 'summary_table.csv'")

    return df_table

# Example usage
if __name__ == "__main__":
    # Replace 'your_file.csv' with your actual CSV filename
    csv_filename = 'llm_as_a_judge_base_llama.csv'

    try:
        results = process_csv_to_json(csv_filename)
    except FileNotFoundError:
        print(f"Error: Could not find '{csv_filename}'")
        print("Please update the filename in the script.")
    except Exception as e:
        print(f"Error processing file: {e}")